# Predictive Models

### Regression Analysis 
- **Predictive Academic Impact** - Modeling how sleep and social media hours predict academic performance.

In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

### Loading our cleaned data

In [2]:
from load_cleaned_data import load_cleaned_data

df = load_cleaned_data()

In [3]:
# Preparing the data
# Select features and target
features = ['sleep_hours', 'daily_social_media_hours']
target = 'academic_performance'

# Convert to numpy (sklearn doesn't work with Polars directly)
X = df.select(features).to_numpy()
y = df.select(target).to_numpy().ravel()

In [4]:
# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# Train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [6]:
# Evaluate
r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R² Score : {r2:.4f}")
print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print()
for feat, coef in zip(features, model.coef_):
    print(f"  {feat:30s} → coefficient: {coef:.4f}")
print(f"  {'Intercept':30s} → {model.intercept_:.4f}")

R² Score : -0.0114
MSE      : 0.3181
RMSE     : 0.5640

  sleep_hours                    → coefficient: 0.0216
  daily_social_media_hours       → coefficient: 0.0038
  Intercept                      → 2.9809


In [7]:
# Visualize — Actual vs Predicted
fig = px.scatter(
    x=y_test,
    y=y_pred,
    labels={'x': 'Actual Academic Performance', 'y': 'Predicted Academic Performance'},
    title='Actual vs Predicted Academic Performance'
)

# Perfect prediction line
fig.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)

fig.show()

In [8]:
# Visualize — Feature Coefficients
coef_df = pl.DataFrame({
    'feature': features,
    'coefficient': model.coef_.tolist()
})

fig2 = px.bar(
    coef_df,
    x='feature',
    y='coefficient',
    color='coefficient',
    color_continuous_scale='RdYlGn',
    title='Feature Coefficients — Impact on Academic Performance'
)
fig2.show()

In [9]:
# Visualize — Residuals
residuals = y_test - y_pred

fig3 = px.scatter(
    x=y_pred,
    y=residuals,
    labels={'x': 'Predicted Values', 'y': 'Residuals'},
    title='Residual Plot'
)

# Zero residual line
fig3.add_hline(y=0, line_dash='dash', line_color='red')
fig3.show()

## Random Forest

In [10]:
# Train Random Forest
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

In [11]:
#Evaluate both models
def evaluate_model(name, y_test, y_pred):
    r2   = r2_score(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    print(f"--- {name} ---")
    print(f"  R²   : {r2:.4f}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}\n")
    return {'Model': name, 'R²': round(r2, 4), 'MSE': round(mse, 4), 'RMSE': round(rmse, 4)}

lr_metrics = evaluate_model('Linear Regression', y_test, y_pred)
rf_metrics = evaluate_model('Random Forest',     y_test, rf_pred)

--- Linear Regression ---
  R²   : -0.0114
  MSE  : 0.3181
  RMSE : 0.5640

--- Random Forest ---
  R²   : -0.3536
  MSE  : 0.4258
  RMSE : 0.6525



In [12]:
# Compare metrics as a table
metrics_df = pl.DataFrame([lr_metrics, rf_metrics])

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['Model', 'R²', 'MSE', 'RMSE'],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='center',
        height=40
    ),
    cells=dict(
        values=[metrics_df[col] for col in metrics_df.columns],
        fill_color=[['#3498db', '#e67e22']],  # blue = LR, orange = RF
        font=dict(color='white', size=12),
        align='center',
        height=35
    )
)])

fig_table.update_layout(title='Model Comparison — Linear Regression vs Random Forest')
fig_table.show()

In [13]:
# Actual vs Predicted - For  both Models
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_test, y=y_pred,
    mode='markers',
    name='Linear Regression',
    marker=dict(color='#3498db', opacity=0.6)
))

fig.add_trace(go.Scatter(
    x=y_test, y=rf_pred,
    mode='markers',
    name='Random Forest',
    marker=dict(color='#e67e22', opacity=0.6)
))

# Perfect prediction line
fig.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)

fig.update_layout(
    title='Actual vs Predicted — Linear Regression vs Random Forest',
    xaxis_title='Actual Academic Performance',
    yaxis_title='Predicted Academic Performance'
)
fig.show()

In [14]:
# Feature Importance - Random Forest Only
importance_df = pl.DataFrame({
    'feature'   : features,
    'importance': rf_model.feature_importances_.tolist()
}).sort('importance', descending=True)

fig2 = px.bar(
    importance_df,
    x='feature',
    y='importance',
    color='importance',
    color_continuous_scale='Oranges',
    title='Random Forest — Feature Importance'
)
fig2.show()

In [15]:
# Residuals - Both Models
fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=y_pred, y=y_test - y_pred,
    mode='markers',
    name='Linear Regression',
    marker=dict(color='#3498db', opacity=0.6)
))

fig3.add_trace(go.Scatter(
    x=rf_pred, y=y_test - rf_pred,
    mode='markers',
    name='Random Forest',
    marker=dict(color='#e67e22', opacity=0.6)
))

fig3.add_hline(y=0, line_dash='dash', line_color='red')

fig3.update_layout(
    title='Residuals — Linear Regression vs Random Forest',
    xaxis_title='Predicted Values',
    yaxis_title='Residuals'
)
fig3.show()